# Forest That Lies on Paper — Green Pixels, Missing Trees

**Standalone.** Synthetic NDVI vs Hansen-like loss; green-but-lost mismatch by month.

VI: Demo synthetic; REAL_DATA = Hansen GFC + MODIS NDVI (VN/SEA).

In [ ]:
%pip install -q numpy pandas matplotlib scipy

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
RNG = np.random.default_rng(7); N, NDVI_THR = 96, 0.6

def make_basemap(n=N):
    yy, xx = np.mgrid[0:n, 0:n]
    forest = np.clip(gaussian_filter(0.75 + 0.1*np.sin(xx/9) + 0.05*RNG.normal(size=(n,n)), 1), 0.1, 0.95)
    agri = (xx > n*0.55) & (xx < n*0.85) & (yy > n*0.2) & (yy < n*0.8)
    loss = np.zeros((n, n), dtype=bool)
    for cy, cx, r in [(30,40,10),(60,70,12),(50,25,8),(70,55,9)]:
        loss |= (yy-cy)**2 + (xx-cx)**2 <= r**2
    loss |= agri & (RNG.random((n,n)) < 0.35)
    return forest, loss, agri

def ndvi_for_month(base, loss, agri, month):
    season = 0.5 + 0.5 * np.sin((month - 3) / 12 * 2 * np.pi)
    ndvi = base.copy()
    natural_loss = loss & ~agri
    ndvi[natural_loss] = 0.25 + 0.05 * RNG.random(natural_loss.sum())
    agri_loss = loss & agri
    ndvi[agri_loss] = 0.35 + 0.45 * season + 0.05 * RNG.random(agri_loss.sum())
    intact = ~loss
    ndvi[intact] = np.clip(base[intact] + 0.05*(season-0.5), 0, 1)
    return np.clip(ndvi, 0, 1)


In [ ]:
base, loss, agri = make_basemap()
rows, ndvi_peak = [], None
for month in range(1, 13):
    ndvi = ndvi_for_month(base, loss, agri, month)
    if month == 8: ndvi_peak = ndvi
    lost = int(loss.sum()); gl = int(((ndvi >= NDVI_THR) & loss).sum())
    rows.append(dict(month=month, n_loss_pixels=lost, n_green_and_lost=gl,
                     green_but_lost_rate=gl/max(lost,1),
                     mean_ndvi_on_loss=float(ndvi[loss].mean()),
                     mean_ndvi_intact=float(ndvi[~loss].mean())))
df = pd.DataFrame(rows); df.to_csv(RESULTS/'mismatch.csv', index=False)
peak = df.loc[df.green_but_lost_rate.idxmax()]
summary = dict(mode='SYNTHETIC', peak_month=int(peak.month),
               peak_green_but_lost_rate=float(peak.green_but_lost_rate),
               mean_rate=float(df.green_but_lost_rate.mean()))
(RESULTS/'summary.json').write_text(json.dumps(summary, indent=2))
print(summary); display(df)


In [ ]:
green_lost = (ndvi_peak >= NDVI_THR) & loss
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
im0 = axes[0].imshow(ndvi_peak, cmap='YlGn', vmin=0, vmax=1)
axes[0].set_title('NDVI peak'); plt.colorbar(im0, ax=axes[0], fraction=0.046)
axes[1].imshow(loss, cmap='Greys'); axes[1].set_title('Loss mask')
axes[2].imshow(green_lost, cmap='magma'); axes[2].set_title('Green ∩ Lost')
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); fig.savefig(RESULTS/'green_vs_loss.png', dpi=120); plt.show()
fig2, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(df.month, df.green_but_lost_rate, marker='o')
ax.set_xlabel('Month'); ax.set_ylabel('Green-but-lost rate'); ax.set_xticks(range(1,13))
fig2.tight_layout(); fig2.savefig(RESULTS/'seasonal_mismatch.png', dpi=120); plt.show()
